In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus()

In [ ]:
import pandas as pd
import plotnine as gg
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import glob
from tqdm import tqdm
import json

import sys

sys.path.append("/workspace/scripts")

from evaluate_distance import evaluate_distances
from essential.utils import PLOTNINE_DEFAULT_THEME_2

In [ ]:
METHODS_TO_PLOT = [
    "fba_moma_default",
    "gene_graph_beta_1",
    "gene_graph_beta_auto",
    "fba_gene_graph_beta_1",
    "fba_gene_graph_beta_auto",
    "LLM",
    "LLM tuned",
]

METHOD_RENAMER = {
    "fba_moma_default": "flux model",
    "gene_graph_beta_1": r"graph ($\beta=1$)",
    "gene_graph_beta_auto": r"graph ($\beta=\lambda_{\max}^{-1}$)",
    "fba_gene_graph_beta_1": r"flux-filt. graph ($\beta=1$)",
    "fba_gene_graph_beta_auto": r"flux-filt. graph ($\beta=\lambda_{\max}^{-1}$)",
    "LLM": "LLM",
    "LLM tuned": "LLM tuned",
}

METHOD_COLORS = {
    "flux model": "#4878CF",  # steel blue — standalone
    r"graph ($\beta=1$)": "#E07B54",  # terracotta
    r"graph ($\beta=\lambda_{\max}^{-1}$)": "#B5503A",  # darker terracotta
    r"flux-filt. graph ($\beta=1$)": "#6AAF6A",  # sage green
    r"flux-filt. graph ($\beta=\lambda_{\max}^{-1}$)": "#3D7A3D",  # darker sage
    "LLM": "#800080",  # purple
    "LLM tuned": "#9932CC",  # lighter/vibrant purple
}

In [ ]:
metabolic_gene_embeddings = pd.read_pickle(
    "experiments/04072026_llm_prior/data/fba_filtered_metabolites_gene_embeddings.pkl"
)

In [ ]:
target_dist = pd.read_pickle("/workspace/results/ecoli_rich_medium/targets/mmd_distances.pkl")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

pca = PCA(n_components=50)
metabolic_gene_embeddings_pca = pca.fit_transform(metabolic_gene_embeddings)

metabolic_dist = pairwise_distances(metabolic_gene_embeddings_pca)
metabolic_dist = pd.DataFrame(
    metabolic_dist, index=metabolic_gene_embeddings.index, columns=metabolic_gene_embeddings.index
)

In [ ]:
genes_to_keep = metabolic_dist.loc["ansA"] >= 1e-4
genes_to_keep = genes_to_keep[genes_to_keep].index
genes_to_keep

In [ ]:
metabolic_dist = metabolic_dist.loc[genes_to_keep].loc[:, genes_to_keep]

In [ ]:
intersect_genes = metabolic_dist.index.intersection(target_dist.index)
metabolic_dist_aligned = metabolic_dist.loc[intersect_genes].loc[:, intersect_genes]
target_dist_aligned = target_dist.loc[intersect_genes].loc[:, intersect_genes]
metabolic_gene_embeddings_aligned = metabolic_gene_embeddings.loc[intersect_genes]

In [ ]:
import jax
import jax.numpy as jnp
import optax
import flax.linen as nn
import pandas as pd


def linear_probe(D, phi, k, lr=0.01, num_steps=1000, seed=42):
    """
    Finds parameters A and b such that psi = phi @ A + b minimizes the squared
    Frobenius norm between its pairwise distance matrix D' and target D.

    Args:
        D: (n, n) jnp.ndarray, target pairwise distance matrix
        phi: (n, d) jnp.ndarray, input features
        k: int, target dimension for psi

    Returns:
        psi_opt: (n, k) jnp.ndarray, optimized transformed features
        loss_df: pd.DataFrame, history of the loss during optimization
    """
    # Define an affine transformation (\psi = \phi A + b)
    model = nn.Dense(features=k)
    key = jax.random.PRNGKey(seed)
    params = model.init(key, phi)

    def loss_fn(params):
        psi = model.apply(params, phi)

        # Compute pairwise distance matrix D' based on \psi
        diffs = psi[:, None, :] - psi[None, :, :]
        D_prime = jnp.sqrt(jnp.sum(diffs**2, axis=-1) + 1e-8)

        # Return squared Frobenius norm ||D - D'||_F^2
        return jnp.sum((D - D_prime) ** 2)
        # return jnp.sum(jnp.abs(D - D_prime))

    # Use Adam optimizer (can be changed to optax.sgd for vanilla GD)
    optimizer = optax.adam(learning_rate=lr)
    opt_state = optimizer.init(params)

    @jax.jit
    def step(params, opt_state):
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    # Optimization loop
    loss_history = []
    for _ in tqdm(range(num_steps)):
        params, opt_state, loss = step(params, opt_state)
        loss_history.append(float(loss))

    # Return the final optimized \psi
    psi_opt = model.apply(params, phi)

    # Create the DataFrame with the loss history
    loss_df = pd.DataFrame({"step": range(num_steps), "loss": loss_history})

    return psi_opt, loss_df

In [ ]:
opt_emb, loss_df = linear_probe(
    metabolic_dist_aligned.values,
    metabolic_gene_embeddings_aligned.values,
    k=128,
    num_steps=10000,
    lr=1e-3,
)

In [ ]:
(
    gg.ggplot(loss_df, gg.aes(x="step", y="loss"))
    + gg.geom_line()
    + gg.theme_minimal()
    + gg.labs(x="Step", y="Loss")
    + gg.scale_y_log10()
)

In [ ]:
from sklearn.manifold import TSNE

reps = TSNE(n_components=2, random_state=42)
opt_emb_pcs = PCA(n_components=50).fit_transform(opt_emb)
metabolic_gene_embeddings_tsne = reps.fit_transform(opt_emb_pcs)

plt.scatter(metabolic_gene_embeddings_tsne[:, 0], metabolic_gene_embeddings_tsne[:, 1])

In [ ]:
dists = opt_emb[:, None, :] - opt_emb[None, :, :]
dists = np.sqrt(np.sum(dists**2, axis=-1))
metabolic_dist_tuned = pd.DataFrame(
    dists, index=metabolic_dist_aligned.index, columns=metabolic_dist_aligned.index
)

In [ ]:
import scipy.stats as stats

stats.spearmanr(metabolic_dist_tuned.values.ravel(), target_dist_aligned.values.ravel())

In [ ]:
stats.spearmanr(metabolic_dist_aligned.values.ravel(), target_dist_aligned.values.ravel())

In [ ]:
plt.scatter(metabolic_dist_tuned.values.ravel(), target_dist_aligned.values.ravel())

# metabolic dists

In [ ]:
distance_metrics_files = glob.glob(
    "/workspace/results/ecoli_rich_medium/*/*/distance_metrics_mmd.json"
)
all_results = []
for f in distance_metrics_files:
    with open(f, "r") as f:
        results = json.load(f)
    all_results.append(results)

In [ ]:
new_results = [
    evaluate_distances(metabolic_dist, target_dist, tag="LLM"),
    evaluate_distances(metabolic_dist_tuned, target_dist, tag="LLM tuned"),
]

In [ ]:
all_results_df = all_results + new_results
all_results_df = pd.DataFrame(all_results_df)

In [ ]:
K_values = [50, 100, 500, 1000]
SELECTED_COLUMNS = [f"target_dist_median_ratio_of_top_{K}_pred_pairs_to_global" for K in K_values]
renamer = {
    f"target_dist_median_ratio_of_top_{K}_pred_pairs_to_global": f"$K={K}$" for K in K_values
}


plot_df = (
    all_results_df.loc[all_results_df["tag"].isin(METHODS_TO_PLOT)]
    .set_index("tag")
    .loc[:, SELECTED_COLUMNS]
)
plot_df = (
    plot_df.loc[plot_df["target_dist_median_ratio_of_top_100_pred_pairs_to_global"].notna()]
    .stack()
    .to_frame("transcriptomic_distance_ratio")
    .reset_index()
    .rename(columns={"level_1": "_K"})
    .assign(
        K=lambda x: pd.Categorical(x["_K"].map(renamer), categories=renamer.values(), ordered=True),
        method=lambda x: x["tag"].map(METHOD_RENAMER),
    )
)

In [ ]:
fig = (
    gg.ggplot(plot_df, gg.aes(x="K", y="transcriptomic_distance_ratio", fill="method"))
    + gg.geom_col(position="dodge")
    + gg.theme_minimal()
    + gg.geom_hline(yintercept=1.0, color="black")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        # legend_position="none",
        axis_text_x=gg.element_text(rotation=45, ha="right", size=5),
        figure_size=(4, 2),
    )
    + gg.scale_fill_manual(values=METHOD_COLORS)
    + gg.labs(x="", y="transcriptomic distance ratio")
)
# fig.save(
#     os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_transcriptomic_distance_ratio.png"), dpi=SAVE_DPI
# )
fig